In [ ]:
# ============================================================
# 🦾 MASTER DASHBOARD: All-In-One (Robust Version)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D 
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import os
import re

# 1. Setup
BASE_LOG_DIR = os.path.join("..", "..", "logs")
# Geometry
L_BASE = 5.0   # cm
L1 = 15.0      # cm
L2 = 15.0      # cm
MAX_ALLOWED_ERROR = 0.5 # cm

# 2. Define the Main Analysis Logic
def load_and_analyze(run_folder_path):
    clear_output(wait=True)
    
    run_name = os.path.basename(run_folder_path)
    is_3d = False
    
    # --- DETECT 3D vs 2D ---
    log_files_3d = glob.glob(os.path.join(run_folder_path, "arm_3d_log.csv"))
    log_files_2d = glob.glob(os.path.join(run_folder_path, "arm_2link_log.csv"))
    
    if log_files_3d:
        df = pd.read_csv(log_files_3d[0])
        is_3d = True
        print(f"📂 Loaded 3D Data: {run_name}")
    elif log_files_2d:
        df = pd.read_csv(log_files_2d[0])
        is_3d = False
        print(f"📂 Loaded 2D Data: {run_name}")
    else:
        print(f"❌ No data found in {run_name}")
        return

    df['time_sec'] = df['timestamp_ms'] / 1000.0
    
    # --- PRE-CALCULATE GEOMETRY (JOINT POSITIONS) ---
    try:
        if is_3d:
            rad_base = np.radians(df['base_deg'])
            rad_shld = np.radians(df['shoulder_deg'])
            r_elbw = L1 * np.cos(rad_shld)
            df['elbow_z'] = L_BASE + L1 * np.sin(rad_shld)
            df['elbow_x'] = r_elbw * np.cos(rad_base)
            df['elbow_y'] = r_elbw * np.sin(rad_base)
        else:
            # Fallback for 2D logs
            if 'motor1_pos' in df.columns:
                rad_m1 = np.radians(df['motor1_pos'])
            else:
                rad_m1 = np.zeros(len(df)) # Safe fallback
                
            df['elbow_x'] = L1 * np.cos(rad_m1)
            df['elbow_y'] = L1 * np.sin(rad_m1)
            df['elbow_z'] = 0 
            df['base_deg'] = 0 
            df['shoulder_deg'] = 0
            df['elbow_deg'] = 0
            
    except Exception as e:
        print(f"⚠️ Warning: Geometry calculation issue: {e}")

    # --- METRICS ---
    duration = df['time_sec'].max()
    max_error = df['error'].max()
    avg_error = df['error'].mean()
    verdict = "PASS" if max_error <= MAX_ALLOWED_ERROR else "FAIL"
    
    # ========================================================
    # 📊 PART 1: STATIC ANALYSIS REPORT (Wrapped in Try/Except)
    # ========================================================
    try:
        fig_static = plt.figure(figsize=(14, 10))
        gs = fig_static.add_gridspec(2, 2, height_ratios=[1.5, 1])
        
        # Plot 1: Cartesian Path
        if is_3d:
            ax1 = fig_static.add_subplot(gs[0, 0], projection='3d')
            ax1.set_title(f"1. 3D Path Reconstructed ({run_name})", fontweight='bold')
            ax1.plot(df['target_x'], df['target_y'], df['target_z'], 'g--', label='Target')
            ax1.plot(df['real_x'], df['real_y'], df['real_z'], 'b-', alpha=0.7, label='Actual')
            ax1.set_zlabel("Z (cm)")
            
            # Skeleton
            last = df.iloc[-1]
            ax1.plot([0, 0, last['elbow_x'], last['real_x']], 
                     [0, 0, last['elbow_y'], last['real_y']], 
                     [0, L_BASE, last['elbow_z'], last['real_z']], 'r-o', lw=3)
        else:
            ax1 = fig_static.add_subplot(gs[0, 0])
            ax1.set_title(f"1. 2D Path ({run_name})", fontweight='bold')
            ax1.plot(df['target_x'], df['target_y'], 'g--', label='Target')
            ax1.plot(df['real_x'], df['real_y'], 'b-', label='Actual')
            ax1.axis('equal')

        ax1.legend()
        ax1.set_xlabel("X (cm)")
        ax1.set_ylabel("Y (cm)")

        # Plot 2: Error Timeline
        ax2 = fig_static.add_subplot(gs[1, :])
        ax2.set_title("2. Tracking Error Over Time", fontweight='bold')
        ax2.plot(df['time_sec'], df['error'], 'r-', label='Error')
        ax2.axhline(MAX_ALLOWED_ERROR, color='k', linestyle=':', label='Limit')
        ax2.fill_between(df['time_sec'], df['error'], color='red', alpha=0.1)
        ax2.set_ylabel("Error (cm)")
        ax2.grid(True, alpha=0.3)
        
        # Text Summary
        ax_text = fig_static.add_subplot(gs[0, 1])
        ax_text.axis('off')
        summary_text = f"""
        RUN REPORT: {run_name}
        -----------------------------
        Type:       {'3D Simulation' if is_3d else '2D Simulation'}
        Duration:   {duration:.2f} s
        VERDICT:    {verdict}
        
        Max Error:  {max_error:.4f} cm
        Avg Error:  {avg_error:.4f} cm
        """
        ax_text.text(0.1, 0.5, summary_text, fontsize=12, family='monospace', va='center')

        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"⚠️ Report Plotting Failed: {e}")
        print("Continuing to Interactive Replay...")

    # ========================================================
    # 🎮 PART 2: 3D INTERACTIVE REPLAY
    # ========================================================
    print("\n👇 3D INTERACTIVE REPLAY (Press Play or Drag Slider) 👇")
    
    def plot_3d_frame(frame_index, elev, azim):
        try:
            fig_3d = plt.figure(figsize=(10, 8))
            ax = fig_3d.add_subplot(111, projection='3d')
            
            if frame_index >= len(df): frame_index = len(df) - 1
            row = df.iloc[frame_index]
            history_slice = df.iloc[:frame_index+1:5]
            
            ax.set_xlim(-20, 30)
            ax.set_ylim(-20, 30)
            ax.set_zlim(0, 30)
            
            info = f"Base: {row['base_deg']:.0f}° | Shldr: {row['shoulder_deg']:.0f}°" if is_3d else "2D Mode"
            ax.set_title(f"T={row['time_sec']:.2f}s | {info}")
            
            # Floor
            ax.plot([-20, 30, 30, -20, -20], [-20, -20, 30, 30, -20], [0,0,0,0,0], 'k-', alpha=0.1) 
            
            # Trace
            z_trace = df['target_z'] if is_3d and 'target_z' in df else np.zeros(len(df))
            z_real = history_slice['real_z'] if is_3d and 'real_z' in df else np.zeros(len(history_slice))
            
            ax.plot(df['target_x'], df['target_y'], z_trace, 'g--', alpha=0.2)
            ax.plot(history_slice['real_x'], history_slice['real_y'], z_real, 'b-', alpha=0.5)

            # Robot Skeleton
            if is_3d:
                joints_x = [0, 0, row['elbow_x'], row['real_x']]
                joints_y = [0, 0, row['elbow_y'], row['real_y']]
                joints_z = [0, L_BASE, row['elbow_z'], row['real_z']]
            else:
                joints_x = [0, row['elbow_x'], row['real_x']]
                joints_y = [0, row['elbow_y'], row['real_y']]
                joints_z = [0, 0, 0]

            ax.plot(joints_x, joints_y, joints_z, 'o-', color='orange', lw=5, markersize=8, markeredgecolor='k')
            ax.plot(joints_x, joints_y, [0]*len(joints_x), 'k-', lw=2, alpha=0.2) # Shadow

            ax.view_init(elev=elev, azim=azim)
            plt.show()
        except Exception as e:
            print(f"Frame Error: {e}")

    # --- WIDGETS ---
    play = widgets.Play(
        value=0, min=0, max=len(df)-1,
        step=10, interval=10, description="Play"
    )
    slider_time = widgets.IntSlider(min=0, max=len(df)-1, step=5, value=0, layout=widgets.Layout(width='60%'))
    widgets.jslink((play, 'value'), (slider_time, 'value'))
    
    slider_elev = widgets.IntSlider(min=0, max=90, step=5, value=30, description='Elev')
    slider_azim = widgets.IntSlider(min=0, max=360, step=5, value=315, description='Azim')
    
    ui = widgets.VBox([
        widgets.HBox([play, slider_time]), 
        widgets.HBox([slider_elev, slider_azim])
    ])
    
    out = widgets.interactive_output(plot_3d_frame, {
        'frame_index': slider_time, 
        'elev': slider_elev, 
        'azim': slider_azim
    })
    
    display(ui, out)

# 3. Main Selection Logic
all_folders = [f.path for f in os.scandir(BASE_LOG_DIR) if f.is_dir()]
arm_folders = sorted(
    [f for f in all_folders if "arm_run" in os.path.basename(f)],
    key=lambda x: int(re.search(r"arm_run_?(\d+)", os.path.basename(x)).group(1)) 
                  if re.search(r"arm_run_?(\d+)", os.path.basename(x)) else 0
)

dashboard_output = widgets.Output()

if not arm_folders:
    print("❌ No 'arm_run' folders found.")
else:
    dropdown = widgets.Dropdown(
        options=[(os.path.basename(f), f) for f in arm_folders],
        value=arm_folders[-1],
        description='Select Run:',
        layout=widgets.Layout(width='50%')
    )
    
    def on_run_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with dashboard_output:
                load_and_analyze(change['new'])

    dropdown.observe(on_run_change)
    display(dropdown, dashboard_output)
    
    # Init
    with dashboard_output:
        load_and_analyze(arm_folders[-1])

Dropdown(description='Select Run:', index=3, layout=Layout(width='50%'), options=(('arm_run_3d_1', '..\\..\\lo…

Output()